In [12]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================
INPUT_FILE = "ALL_ohlcv_long.csv"
OUTPUT_FILE = "engineered_stock_long.csv"
PRICE_COL = "adj_close"

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def compute_rsi(series, window=14):
    """
    Relative Strength Index (RSI)
    """
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def add_features_per_ticker(group):
    """
    Build features for one ticker at a time.
    """
    group = group.sort_values("date").copy()
    price = group[PRICE_COL]

    # -----------------------------------------------------
    # 1. BASE RETURNS
    # -----------------------------------------------------
    # Today's return, computed from adjusted close price
    group["simple_return"] = price.pct_change()

    # Optional log return (kept for reference, not used as main target)
    group["log_return"] = np.log(price / price.shift(1))

    # -----------------------------------------------------
    # 2. TARGET VARIABLE
    # -----------------------------------------------------
    # Tomorrow's return is today's prediction target
    group["target_next_return"] = group["simple_return"].shift(-1)
    group["target_5d_return"] = price.pct_change(5).shift(-5)

    # Optional: implied tomorrow price from tomorrow's return
    group["target_next_price"] = price * (1 + group["target_next_return"])

    # -----------------------------------------------------
    # 3. RETURN LAG FEATURES
    # -----------------------------------------------------
    for lag in [1, 2, 3, 5, 10]:
        group[f"return_lag_{lag}"] = group["simple_return"].shift(lag)

    # -----------------------------------------------------
    # 4. VOLUME LAG FEATURES
    # -----------------------------------------------------
    # Keeping only a couple to avoid redundancy
    for lag in [1, 5]:
        group[f"volume_lag_{lag}"] = group["volume"].shift(lag)

    # -----------------------------------------------------
    # 5. MOMENTUM FEATURES (ROC ONLY)
    # -----------------------------------------------------
    for window in [5, 10, 20]:
        group[f"roc_{window}"] = price.pct_change(window)

    # -----------------------------------------------------
    # 6. VOLATILITY FEATURES
    # -----------------------------------------------------
    # Rolling std of returns = daily volatility proxy
    group["return_std_5"] = group["simple_return"].rolling(5).std()
    group["return_std_20"] = group["simple_return"].rolling(20).std()

    # Exponentially weighted volatility = more weight on recent days
    group["ewm_vol_10"] = group["simple_return"].ewm(span=10, adjust=False).std()

    # -----------------------------------------------------
    # 7. RELATIVE PRICE POSITION
    # -----------------------------------------------------
    # Price relative to recent moving average
    ma_10 = price.rolling(10).mean()
    ma_20 = price.rolling(20).mean()

    group["price_to_sma_10"] = price / ma_10
    group["price_to_sma_20"] = price / ma_20

    # Z-score of price relative to 20-day rolling mean/std
    std_20 = price.rolling(20).std()
    group["zscore_price_20"] = (price - ma_20) / std_20

    # -----------------------------------------------------
    # 8. TECHNICAL INDICATORS
    # -----------------------------------------------------
    group["rsi_14"] = compute_rsi(price, window=14)

    ema_12 = price.ewm(span=12, adjust=False).mean()
    ema_26 = price.ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9, adjust=False).mean()

    # Keep only histogram to reduce redundancy
    group["macd_hist"] = macd - macd_signal

    # -----------------------------------------------------
    # 9. VOLUME FEATURES
    # -----------------------------------------------------
    group["volume_change_1"] = group["volume"].pct_change()

    vol_ma_20 = group["volume"].rolling(20).mean()
    group["relative_volume_20"] = group["volume"] / vol_ma_20

    return group


def add_relative_strength_features(df):
    """
    Add relative strength features using the average return of all stocks
    on a given date as a simple benchmark.
    """
    df = df.copy()

    # Equal-weight benchmark return across all tickers each day
    market_ret = (
        df.groupby("date")["simple_return"]
        .mean()
        .rename("market_return_equal")
        .reset_index()
    )

    df = df.merge(market_ret, on="date", how="left")

    # One-day relative return = stock return minus universe average return
    df["relative_return_1"] = df["simple_return"] - df["market_return_equal"]

    # Rolling relative strength
    for window in [5, 20]:
        stock_roll = df.groupby("ticker")["simple_return"].transform(
            lambda x: x.rolling(window).mean()
        )
        market_roll = df.groupby("ticker")["market_return_equal"].transform(
            lambda x: x.rolling(window).mean()
        )
        df[f"relative_strength_{window}"] = stock_roll - market_roll

    return df


def build_feature_dataset(df):
    """
    Main pipeline for long-format stock data.
    """
    required_cols = ["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    # Per-ticker features
    feat_df = (
        df.groupby("ticker", group_keys=False)
        .apply(add_features_per_ticker)
        .reset_index(drop=True)
    )

    # Cross-sectional relative strength features
    feat_df = feat_df.sort_values(["date", "ticker"]).reset_index(drop=True)
    feat_df = add_relative_strength_features(feat_df)

    return feat_df


def get_feature_columns(df):
    """
    Return the final model feature columns.
    """
    exclude_cols = {
        "ticker",
        "date",
        "log_return",
        "simple_return",        # ← consider excluding
        "target_next_return",
        "target_5d_return",
        "target_next_price",
        "market_return_equal",
    }
    return [col for col in df.columns if col not in exclude_cols]


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    df = pd.read_csv(INPUT_FILE)

    feat_df = build_feature_dataset(df)
    feat_df = feat_df.sort_values(["ticker", "date"]).reset_index(drop=True)

    # Drop rows where target doesn't exist
    feat_df = feat_df.dropna(subset=["target_next_return", "target_5d_return"]).copy()

    feat_df.to_csv(OUTPUT_FILE, index=False)

    feature_cols = get_feature_columns(feat_df)

    print("Feature engineering complete.")
    print(f"Saved to: {OUTPUT_FILE}")
    print(f"Shape: {feat_df.shape}")
    print(f"Number of model features: {len(feature_cols)}")
    print("\nFeature columns:")
    print(feature_cols)

/var/folders/_h/b34996m51sb595dtyp0k7l300000gn/T/ipykernel_9378/1074341618.py:173: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_features_per_ticker)


Feature engineering complete.
Saved to: engineered_stock_long.csv
Shape: (25009, 37)
Number of model features: 29

Feature columns:
['open', 'high', 'low', 'close', 'adj_close', 'volume', 'return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5', 'return_lag_10', 'volume_lag_1', 'volume_lag_5', 'roc_5', 'roc_10', 'roc_20', 'return_std_5', 'return_std_20', 'ewm_vol_10', 'price_to_sma_10', 'price_to_sma_20', 'zscore_price_20', 'rsi_14', 'macd_hist', 'volume_change_1', 'relative_volume_20', 'relative_return_1', 'relative_strength_5', 'relative_strength_20']
